In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Loading dataset...")
df = pd.read_csv("../data/cleaned_arxiv_papers.csv")

print("Loading embeddings...")
embeddings = np.load("../data/arxiv_embeddings.npy")

print("Loading model...")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print(embeddings.shape, embeddings.dtype)

In [ ]:
import faiss
import os

index_path = "../data/paper_faiss.index"
if os.path.exists(index_path):
    print("Loading existing FAISS index...")
    index = faiss.read_index(index_path)
else:
    print("Creating new FAISS index...")
    faiss_embeddings = embeddings.copy()
    faiss.normalize_L2(faiss_embeddings)

    index = faiss.IndexFlatIP(384) 
    index.add(faiss_embeddings)

    os.makedirs("../data", exist_ok=True)
    faiss.write_index(index, index_path)
    print("FAISS index saved successfully!")

print("Papers indexed:", index.ntotal)

In [ ]:
query = "deep learning for medical image analysis"

query_embedding = model.encode([query])
faiss.normalize_L2(query_embedding)

D, I = index.search(query_embedding, 5) 

print("Scores (D):", D)
print("Indices (I):", I)

In [ ]:
print(df.iloc[I[0][0]]['title'])

In [ ]:
def search_paper(query, k=5):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)

    for score, idx in zip(D[0], I[0]):
        print("Similarity Score:", score)
        print("Title:", df.iloc[idx]['title'])
        print("Abstract:", df.iloc[idx]['abstract'][:500])
        print()  

    return D, I

D, I = search_paper(query)

In [ ]:
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6",
    device=0   
)

In [ ]:
sample_summary = summarizer(df.iloc[10466]['abstract'], max_length=120, min_length=40)
print(sample_summary[0]["summary_text"])

In [ ]:
def search_and_summarize(query, k=5):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)

    for score, idx in zip(D[0], I[0]):
        print("Similarity score:", score)
        print("Title:", df.iloc[idx]["title"])
        print("Abstract:", df.iloc[idx]["abstract"][:500])
        print()

        summary = summarizer(df.iloc[idx]["abstract"], max_length=120, min_length=40)
        print("AI SUMMARY:")
        print(summary[0]["summary_text"])
        print("-" * 80)

search_and_summarize("Deep learning in medical science", k=5)

In [ ]:
from keybert import KeyBERT
kw_model = KeyBERT(model=model)
type(kw_model)

In [ ]:
text = df.iloc[26063]['abstract']
keywords = kw_model.extract_keywords(text)
print(keywords)